Import SentiWordNet 3.0.0

In [1]:
!wget https://raw.githubusercontent.com/aesuli/SentiWordNet/master/data/SentiWordNet_3.0.0.txt

--2026-02-11 14:29:13--  https://raw.githubusercontent.com/aesuli/SentiWordNet/master/data/SentiWordNet_3.0.0.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13590743 (13M) [text/plain]
Saving to: ‘SentiWordNet_3.0.0.txt’

SentiWordNet_3.0.0. 100%[===================>]  12.96M  --.-KB/s    in 0.03s   

2026-02-11 14:29:14 (385 MB/s) - ‘SentiWordNet_3.0.0.txt’ saved [13590743/13590743]



Predpríprava SWN - formátovanie

In [2]:
import pandas as pd

file_path = "SentiWordNet_3.0.0.txt"
rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("#"): #ignore lines starting with '#'.
            continue

        parts = line.strip().split("\t") #strip the input line of spaces and split it on indent.
        if len(parts) != 6:
            continue

        pos, sid, pos_score, neg_score, synset_terms, gloss = parts #define parts of each line

        rows.append({ #append each line to a list as a dictionary entry
            "pos": pos,
            "sid": str(sid).zfill(8),
            "pos_score": float(pos_score),
            "neg_score": float(neg_score),
            "obj_score": 1.0 - (float(pos_score) + float(neg_score)), #calculate objective score
            "synset_terms": synset_terms,
            "gloss": gloss
        })

In [3]:
swn_syn = pd.DataFrame(rows) #create a dataframe based on 'rows' list
swn_syn["synset_id"] = swn_syn["pos"] + ":" + swn_syn["sid"] #create a unique id for each synset
swn_syn = swn_syn.drop_duplicates("synset_id").set_index("synset_id")
swn_syn.head()

,pos,sid,pos_score,neg_score,obj_score,synset_terms,gloss
synset_id,,,,,,,
a:00001740,a,00001740,0.125,0.00,0.875,able#1,(usually followed by `to') having the necessar...
a:00002098,a,00002098,0.000,0.75,0.250,unable#1,(usually followed by `to') not having the nece...
a:00002312,a,00002312,0.000,0.00,1.000,dorsal#2 abaxial#1,facing away from the axis of an organ or organ...
a:00002527,a,00002527,0.000,0.00,1.000,ventral#2 adaxial#1,nearest to or facing toward the axis of an org...
a:00002730,a,00002730,0.000,0.00,1.000,acroscopic#1,facing or on the side toward the apex


Import a predpríprava SlovakWordNet

In [6]:
import re

sk_path = "sk-wn-2013-01-23.txt"

rows = []
#find RS block regex
pattern = re.compile(r"␞\s*(\d+)\s+\d+\s+([nvarsa])\b")

with open(sk_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")

        if len(parts) < 4:
            continue

        lex_num, pos_local, lemmas_sk, RS_raw = parts[:4] #define parts of each line
        offset = None
        pos_raw = None

        #find RS marker in each line and assign offset and pos_raw
        for cell in parts:
            m = pattern.search(cell)
            if m:
                offset, pos_raw = m.groups()
                break

        if offset is None:
            continue

        offset = offset.zfill(8)
        pos = "a" if pos_raw == "s" else pos_raw

        rows.append({
            "lex_num": int(lex_num),
            "pos_local": pos_local,
            "lemmas_sk": lemmas_sk,
            "RS_block": RS_raw,
            "pwn_offset": offset,
            "pwn_pos_raw": pos_raw,
            "pwn_pos": pos,
            "synset_id": f"{pos}:{offset}"
        })

sk = pd.DataFrame(rows)
sk.head()

,lex_num,pos_local,lemmas_sk,RS_block,pwn_offset,pwn_pos_raw,pwn_pos,synset_id
0,1999,a,abnormálny; nadmerný; priveľký,␞ 01533535 00 s 01 abnormal 0 001 & 01533120 ...,01533535,s,a,a:01533535
1,31,a,absolútny; oprávnený; zákonný; zákonom ustanovený,␞ 00557108 00 s 01 vested 0 001 & 00556709 a ...,00557108,s,a,a:00557108
2,28,a,absolútny; zvrchovaný,␞ 00719328 00 s 01 absolute 0 002 & 00718924 ...,00719328,s,a,a:00719328
3,2204,a,abstrakčný; schopný abstrahovať; schopný abstr...,␞ 00860932 00 s 01 abstractive 0 003 & 008606...,00860932,s,a,a:00860932
4,35,a,absurdný; nezmyselný,␞ 01431112 00 s 01 absurd 0 003 & 01430847 a ...,01431112,s,a,a:01431112


Mapovanie slovenských lem na SentiWordNet 3.0.0

In [7]:
# split lemmas function
def split_sk_lemmas(x):
    if not isinstance(x, str):
        return []
    out = []
    for t in x.split(";"):
        t = t.strip()
        if t:
            out.append(t.replace(" ", "_"))
    return out

In [8]:
sk["lemmas_sk_list"] = sk["lemmas_sk"].apply(split_sk_lemmas)

#aggregate lemmas
sk_agg = sk.groupby("synset_id")["lemmas_sk_list"].sum().to_frame()
cleaned = []

for lemma in sk_agg["lemmas_sk_list"]:
    cleaned.append(sorted(set(lemma)))
sk_agg["lemmas_sk_list"] = cleaned

#add "#1" sense number
terms = []
for lemmas in sk_agg["lemmas_sk_list"]:
    if not lemmas:
        terms.append(None)
    else:
        s = ""
        for lemma in lemmas:
            s += lemma + "#1 "
        terms.append(s.strip())

sk_agg["synset_terms_sk"] = terms

In [9]:
swn_out = swn_syn.join(sk_agg[["synset_terms_sk"]], how="left")
swn_out["synset_terms_en"] = swn_out["synset_terms"]
#replace where Slovak exists
swn_out["synset_terms"] = swn_out["synset_terms_sk"].fillna(swn_out["synset_terms"])

In [10]:
print("Total SWN synsets:", len(swn_out))
print("Synsets replaced with Slovak terms:", swn_out["synset_terms_sk"].notna().sum())

Total SWN synsets: 117659
Synsets replaced with Slovak terms: 17966


Dataset, ktorý obsahuje len namapované lemy

In [11]:
swn_mapped = (
    swn_out[swn_out["synset_terms_sk"].notna()]
    .drop(columns=["synset_terms_sk", "synset_terms_en"])
    .copy()
)

print("Mapped synsets:", len(swn_mapped))
swn_mapped.head()

Mapped synsets: 17966


,pos,sid,pos_score,neg_score,obj_score,synset_terms,gloss
synset_id,,,,,,,
a:00004980,a,00004980,0.00,0.000,1.000,kompletný#1 neskrátený#1 v_plnom_znení#1 úplný#1,"(used of texts) not shortened; ""an unabridged ..."
a:00005205,a,00005205,0.50,0.000,0.500,absolútny#1 stopercentný#1 úplný#1,"perfect or complete or pure; ""absolute loyalty..."
a:00006032,a,00006032,0.25,0.500,0.250,pomerný#1,estimated by comparison; not absolute or compl...
a:00010726,a,00010726,0.00,0.000,1.000,dravý#1 hltavý#1 nenásytný#1 pažravý#1 vlčí#1 ...,devouring or craving food in great quantities;...
a:00011327,a,00011327,0.00,0.125,0.875,chamtivý#1 hltavý#1 nenásytný#1 prasačí#1 prip...,resembling swine; coarsely gluttonous or greed...


Dataset, ktorý obsahuje nenamapované lemy

In [ ]:
swn_unmapped = (
    swn_out[swn_out["synset_terms_sk"].isna()]
    .drop(columns=["synset_terms_sk", "synset_terms_en"])
    .copy()
)

print("Unmapped synsets:", len(swn_unmapped))
swn_unmapped.head()

Unmapped synsets: 99693


,pos,sid,pos_score,neg_score,obj_score,synset_terms,gloss
synset_id,,,,,,,
a:00001740,a,00001740,0.125,0.00,0.875,able#1,(usually followed by `to') having the necessar...
a:00002098,a,00002098,0.000,0.75,0.250,unable#1,(usually followed by `to') not having the nece...
a:00002312,a,00002312,0.000,0.00,1.000,dorsal#2 abaxial#1,facing away from the axis of an organ or organ...
a:00002527,a,00002527,0.000,0.00,1.000,ventral#2 adaxial#1,nearest to or facing toward the axis of an org...
a:00002730,a,00002730,0.000,0.00,1.000,acroscopic#1,facing or on the side toward the apex


Príprava HelsinkiNLP na preklad nemapovaných lem

In [ ]:
!pip install -q transformers sentencepiece torch

In [ ]:
!pip -q install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 14.9 MB/s eta 0:00:00


In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch

model_name = "Helsinki-NLP/opus-mt-en-sk"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/830k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/302M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(60025, 512, padding_idx=60024)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(60025, 512, padding_idx=60024)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

Funkcie na preklad

In [ ]:
#translate a batch of lemmas
def mt_batch(texts, tokenizer, model, device, batch_size=64, max_length=64, log_every=100):
    outs = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch_id = i // batch_size + 1
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            gen = model.generate(**enc, max_length=max_length)

        outs.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))

        #print progress regularly
        if batch_id % log_every == 0 or batch_id == total_batches:
            print(f"[MT] Batch {batch_id}/{total_batches} completed")

    return outs

#transform, translate, repack
def translate_synset_terms_series(terms_series, tokenizer, model, device, batch_size=64):
    #fill empty and split synset terms
    terms_series = terms_series.fillna("")
    token_lists = terms_series.apply(lambda s: s.split() if s else [])

    lemmas = []
    senses = []
    #remove and save sense numbers
    for toks in token_lists:
        for t in toks:
            if "#" in t:
                w, sn = t.rsplit("#", 1)
            else:
                w, sn = t, "1"
            lemmas.append(w.replace("_", " "))
            senses.append(sn)

    if not lemmas:
        return terms_series.copy()
    #translate batch
    lemmas_sk = mt_batch(lemmas, tokenizer, model, device, batch_size=batch_size, max_length=64)
    lemmas_sk = [x.strip().replace(" ", "_") for x in lemmas_sk]

    out_rows = []
    k = 0
    #readd sense numbers and join into one string
    for toks in token_lists:
        new_toks = []
        for _ in toks:
            new_toks.append(f"{lemmas_sk[k]}#{senses[k]}")
            k += 1
        out_rows.append(" ".join(new_toks))

    return pd.Series(out_rows, index=terms_series.index)

In [ ]:
# translate the unmapped set
swn_unmapped = swn_unmapped.copy()
swn_unmapped["synset_terms_sk_mt"] = translate_synset_terms_series(
    swn_unmapped["synset_terms"],
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=64
)
#replace original synset_terms
swn_unmapped["synset_terms"] = swn_unmapped["synset_terms_sk_mt"]
swn_unmapped = swn_unmapped.drop(columns=["synset_terms_sk_mt"])
swn_unmapped["synset_terms"].head()

[MT] Batch 100/2648 completed
[MT] Batch 200/2648 completed
[MT] Batch 300/2648 completed
[MT] Batch 400/2648 completed
[MT] Batch 500/2648 completed
[MT] Batch 600/2648 completed
[MT] Batch 700/2648 completed
[MT] Batch 800/2648 completed
[MT] Batch 900/2648 completed
[MT] Batch 1000/2648 completed
[MT] Batch 1100/2648 completed
[MT] Batch 1200/2648 completed
[MT] Batch 1300/2648 completed
[MT] Batch 1400/2648 completed
[MT] Batch 1500/2648 completed
[MT] Batch 1600/2648 completed
[MT] Batch 1700/2648 completed
[MT] Batch 1800/2648 completed
[MT] Batch 1900/2648 completed
[MT] Batch 2000/2648 completed
[MT] Batch 2100/2648 completed
[MT] Batch 2200/2648 completed
[MT] Batch 2300/2648 completed
[MT] Batch 2400/2648 completed
[MT] Batch 2500/2648 completed
[MT] Batch 2600/2648 completed
[MT] Batch 2648/2648 completed


,synset_terms
synset_id,
a:00001740,schopné#1
a:00002098,neschopný#1
a:00002312,dorzálne#2 abaxiálne#1
a:00002527,ventrálna#2 adaxiálne#1
a:00002730,akroskopické#1
a:00002843,basecopic#1
a:00002956,únos#1 abducent#1
a:00003131,adduktívna#1 adukcia#1 prídavok#1
a:00003356,pach#1


Merging datasetov mapped a unmapped

In [ ]:
swn_mapped = swn_mapped.copy()
swn_unmapped = swn_unmapped.copy()

swn_mapped["source"] = "sk_wordnet"
swn_unmapped["source"] = "mt"

In [ ]:
swn_final = pd.concat([swn_mapped, swn_unmapped], axis=0)

print("Final rows:", len(swn_final))

swn_final = swn_final.sort_index()
swn_final.head()

Final rows: 117659


,pos,sid,pos_score,neg_score,obj_score,synset_terms,gloss,source
synset_id,,,,,,,,
a:00001740,a,00001740,0.125,0.00,0.875,schopné#1,(usually followed by `to') having the necessar...,mt
a:00002098,a,00002098,0.000,0.75,0.250,neschopný#1,(usually followed by `to') not having the nece...,mt
a:00002312,a,00002312,0.000,0.00,1.000,dorzálne#2 abaxiálne#1,facing away from the axis of an organ or organ...,mt
a:00002527,a,00002527,0.000,0.00,1.000,ventrálna#2 adaxiálne#1,nearest to or facing toward the axis of an org...,mt
a:00002730,a,00002730,0.000,0.00,1.000,akroskopické#1,facing or on the side toward the apex,mt


Kombináciou slovenského WordNetu a preložením zvyšných synsetov SWNu sme dosiahli 100% obsiahnutie synsetov SWN 3.0.0

In [ ]:
print("Duplicates:", swn_final.index.duplicated().sum())
print(swn_final["source"].value_counts())

Duplicates: 0
source
mt            99693
sk_wordnet    17966
Name: count, dtype: int64


Export výsledného lexikónu so zachovaním formátovania SWN 3.0.0

In [ ]:
export_cols = ["pos", "sid", "pos_score", "neg_score", "synset_terms", "gloss"]

#safety checks
missing = [c for c in export_cols if c not in swn_final.columns]
if missing:
    raise ValueError(f"Missing columns in swn_final: {missing}")

out = swn_final.reset_index(drop=True).copy()

#formatting
out["pos"] = out["pos"].astype(str)
out["sid"] = out["sid"].astype(str).str.zfill(8)
out["pos_score"] = out["pos_score"].astype(float)
out["neg_score"] = out["neg_score"].astype(float)

#restore header
header_lines = [
    "# SentiWordNet 3.0.0 – Slovak version",
    "# SynsetTerms localized to Slovak (Slovak WordNet + MT fallback)",
    "# Gloss kept in original English",
    "#",
    "# Format:",
    "# POS\tID\tPosScore\tNegScore\tSynsetTerms\tGloss"
]

#export to txt
out_path = "SentiWordNet_3.0.0_SK.txt"

with open(out_path, "w", encoding="utf-8") as f:
    for line in header_lines:
        f.write(line + "\n")
    out[export_cols].to_csv(f, sep="\t", index=False, header=False)

out_path

'SentiWordNet_3.0.0_SK.txt'